# Hyperparameter-Tuning für XGBoost

## 1. Setup und Konfiguration

In [ ]:
target           = "alighters"
bus_line         = "5"
time_aggregation = "30"
szenario = "1"

In [31]:
use_sampling     = True     # True = nur 10 % der Trainingsdaten, False = gesamte 80 %
n_trials         = 100     # Anzahl der Random-Search-Trials
n_splits         = 5       # Anzahl der Folds für TimeSeriesSplit
train_frac       = 0.8     # Split-Verhältnis Train/Test über gesamten Datensatz

## 2. Import relevanter Bibliotheken

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from joblib import Parallel, delayed
from xgboost import XGBRegressor
import numpy as np
import pandas as pd
import time

## 3. Datenbeschaffung mit Spark

In [32]:
# Spark-Session & Daten laden (nur Filterung)
spark = SparkSession.builder.getOrCreate()
df_raw = (
    spark.read
         .parquet(
             "data/enriched_data"
         )
         .filter(
             (col("line") == bus_line) &
             (col("covid_period") == "Post-COVID")
         )
)

## 4. Feature-Auswahl

In [33]:
# Wichtige Feature-Spalten auswählen

if target == "occupancy_s-1":
    dynamic_feats = [
        f"interval_{time_aggregation}min",
        f"bus_count_interval_{time_aggregation}min",
        f"avg_occupancy_s-1_interval_{time_aggregation}min",
        f"lag_occupancy_s-1_interval_{time_aggregation}min"
    ]
else:
    dynamic_feats = [
        f"interval_{time_aggregation}min",
        f"bus_count_interval_{time_aggregation}min",
        f"avg_{target}_interval_{time_aggregation}min",
        f"lag_{target}_interval_{time_aggregation}min",
        "deviation","stop_skipped"
    ]
static_feats = [
    "weekday","month","is_day","temperature_2m","rain",
    "weighted_avg_rain","snowfall","wind_speed_10m"
]
cat_cols = [
    "stop_name","line","direction","lag_stop_name","lead_stop_name",
    "rain_bins","event_type","is_public_holiday","is_school_holiday"
]

# Hinzufügen weiterer Daten im zweiten Szenario
all_features = cat_cols + static_feats + dynamic_feats + [target] + (["occupancy_s-2", "delay_s-2"] if szenario == "2" and target == "occupancy_s-1" else [])

## 5. Datenaufbereitung

In [ ]:
# Spark → Pandas konvertieren und sortieren
pd_df = df_raw.orderBy("date", "actual_arrival").select(*all_features).toPandas()

# Korrektes Datentyp-Setzen für native Kategoriensupport
for c in cat_cols:
    pd_df[c] = pd_df[c].astype("category")

# Train/Test-Split
cutoff = int(len(pd_df) * train_frac)
train_df = pd_df.iloc[:cutoff]
test_df  = pd_df.iloc[cutoff:]

if use_sampling:
    train_df = train_df.sample(frac=0.1, random_state=42)

X_train_full = train_df[cat_cols + static_feats + dynamic_feats]
y_train_full = train_df[target].values

X_test = test_df[cat_cols + static_feats + dynamic_feats]
y_test = test_df[target].values

## 6. Cross-Validation Setup

In [ ]:
# TimeSeriesSplit CV vorbereiten
tscv = TimeSeriesSplit(n_splits=n_splits)

## 7. Hyperparameter-Suchraum

In [35]:
# 7️⃣ Suchraum definieren
param_dist = {
    "n_estimators":     (100,   1000),
    "max_depth":        (3,     12),
    "learning_rate":    (0.01,  0.3),      # log-uniform
    "subsample":        (0.5,   1.0),
    "colsample_bytree": (0.5,   1.0),
    "gamma":            (0.0,   5.0),
    "reg_alpha":        (1e-3,  1.0),      # log-uniform
    "reg_lambda":       (0.0,   1.0),
    "min_child_weight": (1,     10)        # integer
}

def sample_params():
    return {
        "n_estimators":     int(np.random.randint(*param_dist["n_estimators"])),
        "max_depth":        int(np.random.randint(*param_dist["max_depth"])),
        "min_child_weight": int(np.random.randint(*param_dist["min_child_weight"])),
        "learning_rate":    10 ** np.random.uniform(
                                  np.log10(param_dist["learning_rate"][0]),
                                  np.log10(param_dist["learning_rate"][1])
                              ),
        "reg_alpha":        10 ** np.random.uniform(
                                  np.log10(param_dist["reg_alpha"][0]),
                                  np.log10(param_dist["reg_alpha"][1])
                              ),
        "subsample":        float(np.random.uniform(*param_dist["subsample"])),
        "colsample_bytree": float(np.random.uniform(*param_dist["colsample_bytree"])),
        "gamma":            float(np.random.uniform(*param_dist["gamma"])),
        "reg_lambda":       float(np.random.uniform(*param_dist["reg_lambda"]))
    }

random_combinations = [sample_params() for _ in range(n_trials)]

## 8. Evaluationsfunktion

In [36]:
# Evaluations-Funktion (CV)
def eval_params(params):
    rmses = []
    for train_idx, val_idx in tscv.split(X_train_full):
        X_tr = X_train_full.iloc[train_idx]
        X_val = X_train_full.iloc[val_idx]
        y_tr = y_train_full[train_idx]
        y_val = y_train_full[val_idx]

        model = XGBRegressor(
            objective="reg:squarederror",
            tree_method="hist",
            enable_categorical=True,
            random_state=42,
            n_jobs=1,
            **params
        )
        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)
        rmses.append(mean_squared_error(y_val, preds, squared=False))
    return np.mean(rmses), params


## 9. Randomized Search mit Parallelisierung

In [ ]:
# Parallele Random Search mit Joblib
start_time = time.time()
print("▶ Starte parallelisierten Random Search mit enable_categorical …")
results = Parallel(n_jobs=-1, verbose=10)(
    delayed(eval_params)(p) for p in random_combinations
)
best_rmse, best_params = min(results, key=lambda x: x[0])
search_duration = time.time() - start_time
print(f"⚙️ Beste Params: {best_params} | CV-RMSE={best_rmse:.4f} in {search_duration:.1f}s")

## 10. Finales Modelltraining und Testauswertung

In [38]:
# Finales Modell trainieren & testen
final_model = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    enable_categorical=True,
    random_state=42,
    n_jobs=-1,
    **best_params
)
final_model.fit(X_train_full, y_train_full)
preds_test = final_model.predict(X_test)
rmse_test = mean_squared_error(y_test, preds_test, squared=False)
print(f"🏁 Test-RMSE: {rmse_test:.4f}")

## 11. Speicherung der Ergebnisse

In [39]:
# Export der besten Modelparameter 
results_df = pd.DataFrame({
    "algorithm":    ["XGBoost_TSCV_EnableCategorical"],
    "search_time_s": [search_duration],
    "cv_rmse":      [best_rmse],
    "test_rmse":    [rmse_test],
    "best_params":  [str(best_params)]
})
BASE_PATH = (
    "machine_learning/hyperparameter_tuning/optimal_parameters/"
)

out_path = f"{BASE_PATH}/machine_learning/hyperparameter_tuning/optimal_parameters/hyperparameter_tuning_szenario{szenario}/XGB_RS_{bus_line}_{target}_{time_aggregation}_szenario{str(szenario)}.csv"

print(f"▶ Speichere Ergebnisse nach:\n  {out_path}")
results_df.to_csv(out_path, index=False)